
# Generic Health Insurance EDA & Pattern Rule Mining — File or Databricks Table

This notebook supports binary, one-vs-rest, multiclass, and ordinal targets, and can read from:

- Uploaded Excel, CSV, or Parquet files
- Databricks Unity Catalog or Hive Metastore tables
- SQL queries over Delta tables or views

For high-volume sources, the notebook keeps aggregation and feature preparation in Spark, limits driver collection, samples only where algorithms require pandas/scikit-learn, prunes high-cardinality features, and writes results to Delta or downloadable files.

> Discovered patterns are associations. Validate for leakage, stability, fairness, medical plausibility, and operational usefulness before deployment.


## 1. Install and import libraries

In [ ]:

# Databricks already includes PySpark. Install only lightweight libraries not present in the runtime.
# Restart Python only when Databricks requests it.
%pip install -q mlxtend openpyxl


In [ ]:

import os, re, math, warnings, json
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, export_text, _tree
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
from mlxtend.frequent_patterns import fpgrowth, association_rules

# Spark imports are available on Databricks. The fallback keeps file-only local execution possible.
try:
    from pyspark.sql import functions as F, types as T, Window
    from pyspark.storagelevel import StorageLevel
    SPARK_AVAILABLE = True
except Exception:
    SPARK_AVAILABLE = False


## 2. Configuration

In [ ]:

# ----------------------------- SOURCE CONFIGURATION -----------------------------
# FILE: pandas for modest files; Spark for CSV/Parquet when USE_SPARK_FOR_FILE=True.
# TABLE: spark.table(TABLE_NAME)
# SQL: spark.sql(SQL_QUERY)
SOURCE_TYPE = "TABLE"                 # FILE, TABLE, or SQL

FILE_PATH = "/Volumes/catalog/schema/volume/data.parquet"
FILE_FORMAT = "parquet"               # parquet, csv, excel
SHEET_NAME = 0
CSV_OPTIONS = {"header": "true", "inferSchema": "true"}
USE_SPARK_FOR_FILE = True              # Excel always uses pandas

TABLE_NAME = "catalog.schema.claims_table"
SQL_QUERY = "SELECT * FROM catalog.schema.claims_table"
SOURCE_FILTER = None                   # Example: "claim_date >= date_sub(current_date(), 730)"
SOURCE_COLUMNS = None                  # Explicit projection is strongly recommended

# ----------------------------- TARGET CONFIGURATION -----------------------------
TARGET_COLUMN = "risk_level"
TARGET_MODE = "multiclass"            # binary, one_vs_rest, multiclass, ordinal
POSITIVE_CLASS = "Y"
FOCUS_CLASS = "High"
TARGET_CLASSES = ["Low", "Medium", "High"]
TARGET_ORDER = {"Low": 1, "Medium": 2, "High": 3}

ID_COLUMNS = []
DATE_COLUMNS = []
FREE_TEXT_COLUMNS = []
FORCE_CATEGORICAL = []
FORCE_NUMERIC = []
EXCLUDE_COLUMNS = []

HEALTH_FIELD_MAP = {
    "claim_amount": None, "premium_amount": None, "sum_insured": None,
    "policy_start_date": None, "claim_date": None,
    "admission_date": None, "discharge_date": None,
    "member_age": None, "hospital_id": None, "provider_city": None,
    "diagnosis_code": None, "procedure_code": None, "room_rent": None,
    "length_of_stay": None, "network_flag": None, "cashless_flag": None,
    "claim_type": None,
}

# ----------------------------- SCALE / COST CONTROLS -----------------------------
# Spark EDA runs over the complete source. Driver-side ML/rule mining uses a bounded sample.
MAX_DRIVER_ROWS = 250_000
SAMPLE_FRACTION = None                 # None = automatically estimated from count
STRATIFIED_SAMPLE = True
SAMPLE_SEED = 42

# Avoid repeated full scans. Persist only the projected/filtered/feature-engineered frame.
ENABLE_CACHE = True
CACHE_STORAGE_LEVEL = "DISK_ONLY"      # DISK_ONLY is safer and cheaper for large datasets
UNPERSIST_AT_END = True

# High-cardinality controls
MAX_CATEGORICAL_CARDINALITY = 100
TOP_N_CATEGORIES = 30
MAX_MODEL_CATEGORIES = 50
DROP_NEAR_UNIQUE_PCT = 0.95

# Rule-search controls. Exhaustive search grows combinatorially.
MIN_SUPPORT_COUNT = 100
MIN_CLASS_COUNT = 20
MIN_CONFIDENCE = 0.35
MIN_LIFT = 1.30
MAX_RULE_LENGTH = 3
MAX_RULE_FEATURES = 20                 # rank features first, then mine only top features
MAX_NUMERIC_BINS = 5
MAX_RULES_PER_LEVEL = 5_000

# FP-Growth on pandas sample only
RUN_ASSOCIATION_RULES = True
ASSOCIATION_MIN_SUPPORT = 0.02
ASSOCIATION_MIN_CONFIDENCE = 0.35
ASSOCIATION_MIN_LIFT = 1.30
ASSOCIATION_MAX_LEN = 4
MAX_ASSOCIATION_COLUMNS = 20

# Decision tree controls
RUN_DECISION_TREE = True
TREE_MAX_DEPTH = 5
TREE_MIN_SAMPLES_LEAF = 100
TEST_SIZE = 0.25
RANDOM_STATE = 42

# Output: use Delta for large result sets and Excel/CSV for compact downloadable summaries.
OUTPUT_MODE = "BOTH"                  # DELTA, FILES, BOTH
OUTPUT_FOLDER = "/dbfs/FileStore/pattern_mining_outputs"
OUTPUT_EXCEL = "health_insurance_pattern_rules.xlsx"
OUTPUT_DELTA_SCHEMA = "catalog.schema"
OUTPUT_TABLE_PREFIX = "pattern_mining"


## 3. Load source with projection, filtering, and Spark-first execution

In [ ]:

def clean_name(c):
    return re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_")

def normalize_config_names():
    global TARGET_COLUMN, ID_COLUMNS, DATE_COLUMNS, FREE_TEXT_COLUMNS
    global FORCE_CATEGORICAL, FORCE_NUMERIC, EXCLUDE_COLUMNS, SOURCE_COLUMNS
    TARGET_COLUMN = clean_name(TARGET_COLUMN)
    ID_COLUMNS = [clean_name(c) for c in ID_COLUMNS]
    DATE_COLUMNS = [clean_name(c) for c in DATE_COLUMNS]
    FREE_TEXT_COLUMNS = [clean_name(c) for c in FREE_TEXT_COLUMNS]
    FORCE_CATEGORICAL = [clean_name(c) for c in FORCE_CATEGORICAL]
    FORCE_NUMERIC = [clean_name(c) for c in FORCE_NUMERIC]
    EXCLUDE_COLUMNS = [clean_name(c) for c in EXCLUDE_COLUMNS]
    if SOURCE_COLUMNS:
        SOURCE_COLUMNS = [clean_name(c) for c in SOURCE_COLUMNS]
    for k, v in list(HEALTH_FIELD_MAP.items()):
        if v:
            HEALTH_FIELD_MAP[k] = clean_name(v)

normalize_config_names()

def load_source():
    source = SOURCE_TYPE.upper()
    if source == "TABLE":
        if not SPARK_AVAILABLE:
            raise RuntimeError("TABLE source requires Databricks/Spark.")
        sdf = spark.table(TABLE_NAME)
    elif source == "SQL":
        if not SPARK_AVAILABLE:
            raise RuntimeError("SQL source requires Databricks/Spark.")
        sdf = spark.sql(SQL_QUERY)
    elif source == "FILE":
        fmt = FILE_FORMAT.lower()
        if fmt == "excel":
            pdf = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
            pdf.columns = [clean_name(c) for c in pdf.columns]
            return None, pdf
        if USE_SPARK_FOR_FILE and SPARK_AVAILABLE:
            if fmt == "csv":
                sdf = spark.read.options(**CSV_OPTIONS).csv(FILE_PATH)
            elif fmt == "parquet":
                sdf = spark.read.parquet(FILE_PATH)
            else:
                raise ValueError(f"Unsupported Spark file format: {fmt}")
        else:
            if fmt == "csv":
                pdf = pd.read_csv(FILE_PATH)
            elif fmt == "parquet":
                pdf = pd.read_parquet(FILE_PATH)
            else:
                raise ValueError(f"Unsupported file format: {fmt}")
            pdf.columns = [clean_name(c) for c in pdf.columns]
            return None, pdf
    else:
        raise ValueError("SOURCE_TYPE must be FILE, TABLE, or SQL")

    # Standardize Spark column names before projection.
    sdf = sdf.toDF(*[clean_name(c) for c in sdf.columns])
    if SOURCE_COLUMNS:
        missing = sorted(set(SOURCE_COLUMNS) - set(sdf.columns))
        if missing:
            raise KeyError(f"SOURCE_COLUMNS not present: {missing}")
        sdf = sdf.select(*SOURCE_COLUMNS)
    if SOURCE_FILTER:
        sdf = sdf.where(SOURCE_FILTER)
    return sdf, None

sdf, df = load_source()

if sdf is not None:
    print("Spark source loaded. Columns:", len(sdf.columns))
    display(sdf.limit(10))
else:
    print("Pandas source loaded. Shape:", df.shape)
    display(df.head(10))


## 4. Spark-aware profiling, feature engineering, and target preparation

In [ ]:

MISSING_STRINGS = ["?", "", "NA", "N/A", "NULL", "null", "None", "none", "-", "--"]

def add_spark_features(sdf):
    # Normalize configured date fields and missing string tokens without Python UDFs.
    for c in DATE_COLUMNS:
        if c in sdf.columns:
            sdf = sdf.withColumn(c, F.to_timestamp(F.col(c)))
    for field in sdf.schema.fields:
        if isinstance(field.dataType, T.StringType):
            c = field.name
            sdf = sdf.withColumn(
                c,
                F.when(F.trim(F.col(c)).isin(MISSING_STRINGS), F.lit(None))
                 .otherwise(F.trim(F.col(c)))
            )

    def has(key):
        c = HEALTH_FIELD_MAP.get(key)
        return bool(c and c in sdf.columns)

    if has("claim_amount") and has("premium_amount"):
        sdf = sdf.withColumn(
            "claim_to_premium_ratio",
            F.when(F.col(HEALTH_FIELD_MAP["premium_amount"]) != 0,
                   F.col(HEALTH_FIELD_MAP["claim_amount"]).cast("double") /
                   F.col(HEALTH_FIELD_MAP["premium_amount"]).cast("double"))
        )
    if has("claim_amount") and has("sum_insured"):
        sdf = sdf.withColumn(
            "claim_to_sum_insured_ratio",
            F.when(F.col(HEALTH_FIELD_MAP["sum_insured"]) != 0,
                   F.col(HEALTH_FIELD_MAP["claim_amount"]).cast("double") /
                   F.col(HEALTH_FIELD_MAP["sum_insured"]).cast("double"))
        )
    if has("policy_start_date") and has("claim_date"):
        sdf = sdf.withColumn(
            "policy_tenure_days_at_claim",
            F.datediff(F.to_date(F.col(HEALTH_FIELD_MAP["claim_date"])),
                       F.to_date(F.col(HEALTH_FIELD_MAP["policy_start_date"])))
        )
    if has("admission_date") and has("discharge_date"):
        sdf = sdf.withColumn(
            "calculated_length_of_stay",
            F.datediff(F.to_date(F.col(HEALTH_FIELD_MAP["discharge_date"])),
                       F.to_date(F.col(HEALTH_FIELD_MAP["admission_date"])))
        )
    return sdf

def prepare_spark_target(sdf):
    sdf = sdf.where(F.col(TARGET_COLUMN).isNotNull())
    mode = TARGET_MODE.lower()
    normalized = F.lower(F.trim(F.col(TARGET_COLUMN).cast("string")))
    if mode == "binary":
        sdf = sdf.withColumn("target_flag", (normalized == str(POSITIVE_CLASS).strip().lower()).cast("int"))
    elif mode == "one_vs_rest":
        sdf = sdf.withColumn("target_flag", (normalized == str(FOCUS_CLASS).strip().lower()).cast("int"))
    elif mode == "multiclass":
        sdf = sdf.withColumn("target_class", F.trim(F.col(TARGET_COLUMN).cast("string")))
    elif mode == "ordinal":
        mapping_expr = F.create_map(*sum(([F.lit(str(k).strip().lower()), F.lit(float(v))] for k,v in TARGET_ORDER.items()), []))
        sdf = sdf.withColumn("target_class", F.trim(F.col(TARGET_COLUMN).cast("string")))
        sdf = sdf.withColumn("target_ordinal", mapping_expr[normalized]).where(F.col("target_ordinal").isNotNull())
    else:
        raise ValueError("Invalid TARGET_MODE")
    return sdf

def pandas_prepare(pdf):
    pdf = pdf.copy()
    pdf = pdf.replace(MISSING_STRINGS, np.nan)
    for c in pdf.select_dtypes(include="object").columns:
        pdf[c] = pdf[c].astype("string").str.strip()
    for c in DATE_COLUMNS:
        if c in pdf.columns:
            pdf[c] = pd.to_datetime(pdf[c], errors="coerce")
    pdf = pdf[pdf[TARGET_COLUMN].notna()].copy()
    norm = pdf[TARGET_COLUMN].astype(str).str.strip().str.lower()
    if TARGET_MODE == "binary":
        pdf["target_flag"] = (norm == str(POSITIVE_CLASS).strip().lower()).astype(int)
    elif TARGET_MODE == "one_vs_rest":
        pdf["target_flag"] = (norm == str(FOCUS_CLASS).strip().lower()).astype(int)
    elif TARGET_MODE == "multiclass":
        pdf["target_class"] = pdf[TARGET_COLUMN].astype(str).str.strip()
    else:
        order = {str(k).strip().lower(): v for k,v in TARGET_ORDER.items()}
        pdf["target_class"] = pdf[TARGET_COLUMN].astype(str).str.strip()
        pdf["target_ordinal"] = norm.map(order)
        pdf = pdf[pdf["target_ordinal"].notna()].copy()
    return pdf

if sdf is not None:
    sdf = prepare_spark_target(add_spark_features(sdf))
    if ENABLE_CACHE:
        level = getattr(StorageLevel, CACHE_STORAGE_LEVEL, StorageLevel.DISK_ONLY)
        sdf.persist(level)
    row_count = sdf.count()  # one controlled materialization; reused for sampling and profile rates
    print("Filtered row count:", row_count)
else:
    df = pandas_prepare(df)
    row_count = len(df)
    print("Row count:", row_count)


## 5. Feature typing, cardinality pruning, and bounded sampling

In [ ]:

def spark_feature_types(sdf):
    excluded = set([TARGET_COLUMN, "target_flag", "target_class", "target_ordinal"] + ID_COLUMNS + FREE_TEXT_COLUMNS + EXCLUDE_COLUMNS)
    numeric, categorical, dates = [], [], []
    for f in sdf.schema.fields:
        c = f.name
        if c in excluded:
            continue
        if c in FORCE_CATEGORICAL:
            categorical.append(c)
        elif c in FORCE_NUMERIC:
            numeric.append(c)
        elif isinstance(f.dataType, (T.ByteType,T.ShortType,T.IntegerType,T.LongType,T.FloatType,T.DoubleType,T.DecimalType)):
            numeric.append(c)
        elif isinstance(f.dataType, (T.DateType,T.TimestampType)):
            dates.append(c)
        else:
            categorical.append(c)
    return numeric, categorical, dates

def pandas_feature_types(pdf):
    excluded=set([TARGET_COLUMN,"target_flag","target_class","target_ordinal"]+ID_COLUMNS+FREE_TEXT_COLUMNS+EXCLUDE_COLUMNS)
    numeric=[]; categorical=[]; dates=[]
    for c in pdf.columns:
        if c in excluded: continue
        if c in FORCE_CATEGORICAL: categorical.append(c)
        elif c in FORCE_NUMERIC: numeric.append(c)
        elif pd.api.types.is_datetime64_any_dtype(pdf[c]): dates.append(c)
        elif pd.api.types.is_numeric_dtype(pdf[c]): numeric.append(c)
        else: categorical.append(c)
    return numeric,categorical,dates

if sdf is not None:
    numeric_cols, categorical_cols, date_cols = spark_feature_types(sdf)

    # Approximate cardinality in one aggregation rather than one scan per column.
    card_exprs = [F.approx_count_distinct(F.col(c)).alias(c) for c in categorical_cols]
    cardinality = sdf.agg(*card_exprs).first().asDict() if card_exprs else {}
    categorical_cols = [c for c in categorical_cols if cardinality.get(c,0) <= MAX_CATEGORICAL_CARDINALITY]

    # Bounded sample for driver-only algorithms. Stratify for classification targets where possible.
    fraction = SAMPLE_FRACTION or min(1.0, MAX_DRIVER_ROWS / max(row_count,1))
    target_for_sampling = "target_flag" if TARGET_MODE in ["binary","one_vs_rest"] else "target_class"
    if STRATIFIED_SAMPLE and fraction < 1.0:
        class_values = [r[0] for r in sdf.select(target_for_sampling).distinct().collect()]
        fractions = {v: fraction for v in class_values}
        sample_sdf = sdf.sampleBy(target_for_sampling, fractions=fractions, seed=SAMPLE_SEED)
    else:
        sample_sdf = sdf.sample(False, fraction, SAMPLE_SEED) if fraction < 1.0 else sdf

    sample_cols = list(dict.fromkeys(
        numeric_cols + categorical_cols + [TARGET_COLUMN] +
        (["target_flag"] if TARGET_MODE in ["binary","one_vs_rest"] else ["target_class"]) +
        (["target_ordinal"] if TARGET_MODE == "ordinal" else [])
    ))
    df = sample_sdf.select(*sample_cols).limit(MAX_DRIVER_ROWS).toPandas()
    print("Driver sample rows:", len(df), "fraction:", round(fraction,6))
else:
    numeric_cols, categorical_cols, date_cols = pandas_feature_types(df)
    if len(df) > MAX_DRIVER_ROWS:
        target_for_sampling = "target_flag" if TARGET_MODE in ["binary","one_vs_rest"] else "target_class"
        df, _ = train_test_split(df, train_size=MAX_DRIVER_ROWS, stratify=df[target_for_sampling], random_state=SAMPLE_SEED)

print("Numeric features:", len(numeric_cols))
print("Categorical features retained:", len(categorical_cols))


## 6. Full-volume Spark EDA and compact driver summaries

In [ ]:

def full_volume_profile_spark(sdf, numeric_cols, categorical_cols):
    # Compact column profile. Approximate distinct counts reduce shuffle cost.
    rows=[]
    total=row_count
    for batch_start in range(0, len(sdf.columns), 40):
        batch=sdf.columns[batch_start:batch_start+40]
        exprs=[]
        for c in batch:
            exprs += [
                F.sum(F.col(c).isNull().cast("long")).alias(f"{c}__missing"),
                F.approx_count_distinct(F.col(c)).alias(f"{c}__distinct")
            ]
        values=sdf.agg(*exprs).first().asDict()
        for c in batch:
            rows.append({"column":c,"rows":total,"missing":values[f"{c}__missing"],
                         "missing_pct":values[f"{c}__missing"]/max(total,1),
                         "approx_distinct":values[f"{c}__distinct"]})
    return pd.DataFrame(rows)

if sdf is not None:
    profile = full_volume_profile_spark(sdf, numeric_cols, categorical_cols)

    # Numeric summaries are computed in one Spark aggregation.
    num_expr=[]
    for c in numeric_cols:
        num_expr += [F.count(c).alias(f"{c}__count"), F.mean(c).alias(f"{c}__mean"),
                     F.min(c).alias(f"{c}__min"), F.max(c).alias(f"{c}__max"),
                     F.expr(f"percentile_approx(`{c}`, 0.5, 10000)").alias(f"{c}__median")]
    vals=sdf.agg(*num_expr).first().asDict() if num_expr else {}
    numeric_summary=pd.DataFrame([{ "feature":c, "count":vals.get(f"{c}__count"),
        "mean":vals.get(f"{c}__mean"), "min":vals.get(f"{c}__min"),
        "median":vals.get(f"{c}__median"), "max":vals.get(f"{c}__max")}
        for c in numeric_cols])
else:
    profile=pd.DataFrame({"column":df.columns,"rows":len(df),
                          "missing":[df[c].isna().sum() for c in df.columns],
                          "missing_pct":[df[c].isna().mean() for c in df.columns],
                          "approx_distinct":[df[c].nunique(dropna=True) for c in df.columns]})
    numeric_summary=df[numeric_cols].describe(percentiles=[.5]).T.reset_index().rename(columns={"index":"feature","50%":"median"}) if numeric_cols else pd.DataFrame()

display(profile.sort_values("missing_pct",ascending=False).head(100))
display(numeric_summary.head(100))


## 7. Feature ranking, bounded binning, and rule-mining frame

In [ ]:

def discretize_sample(pdf, cols, max_bins=5):
    out=pd.DataFrame(index=pdf.index)
    meta=[]
    for c in cols:
        s=pd.to_numeric(pdf[c],errors="coerce")
        if s.nunique(dropna=True)<2: continue
        try:
            b,e=pd.qcut(s,q=min(max_bins,s.nunique()),duplicates="drop",retbins=True)
            out[f"{c}__bin"]=b.astype("string").fillna("UNKNOWN")
            meta.append({"feature":c,"edges":json.dumps(e.tolist())})
        except Exception:
            pass
    return out,pd.DataFrame(meta)

# Rank categorical features by maximum class lift using grouped sample aggregations.
def feature_lift_score(pdf, feature):
    t="target_flag" if TARGET_MODE in ["binary","one_vs_rest"] else "target_class"
    x=pdf[[feature,t]].copy(); x[feature]=x[feature].astype("string").fillna("UNKNOWN")
    if TARGET_MODE in ["binary","one_vs_rest"]:
        g=x.groupby(feature)[t].agg(["count","sum"]); base=x[t].mean()
        return float((g["sum"]/g["count"]/max(base,1e-9)).replace([np.inf,-np.inf],np.nan).max())
    rates=x[t].value_counts(normalize=True)
    g=x.groupby([feature,t]).size().rename("n").reset_index()
    support=x.groupby(feature).size().rename("total").reset_index()
    g=g.merge(support,on=feature); g["lift"]=(g["n"]/g["total"])/g[t].map(rates)
    return float(g["lift"].replace([np.inf,-np.inf],np.nan).max())

candidate_cat=[]
for c in categorical_cols:
    if c in df.columns:
        n=df[c].nunique(dropna=True)
        if 1 < n <= MAX_CATEGORICAL_CARDINALITY and n/max(len(df),1) < DROP_NEAR_UNIQUE_PCT:
            candidate_cat.append(c)

scores=[]
for c in candidate_cat:
    try: scores.append((c,feature_lift_score(df,c)))
    except Exception: pass
selected_cat=[c for c,_ in sorted(scores,key=lambda z:(z[1] if pd.notna(z[1]) else -1),reverse=True)[:MAX_RULE_FEATURES]]

# Numeric features enter rule mining as quantile bins; cap total rule features.
selected_num=numeric_cols[:max(0,MAX_RULE_FEATURES-len(selected_cat))]
numeric_binned,bin_metadata=discretize_sample(df,selected_num,MAX_NUMERIC_BINS)
rule_frame=pd.DataFrame(index=df.index)
for c in selected_cat:
    v=df[c].astype("string").fillna("UNKNOWN")
    top=v.value_counts().head(TOP_N_CATEGORIES).index
    rule_frame[c]=v.where(v.isin(top),"OTHER")
rule_frame=pd.concat([rule_frame,numeric_binned],axis=1)
if TARGET_MODE in ["binary","one_vs_rest"]:
    rule_frame["target_flag"]=df["target_flag"].values
else:
    rule_frame["target_class"]=df["target_class"].astype(str).values
    if TARGET_MODE=="ordinal": rule_frame["target_ordinal"]=df["target_ordinal"].values
print("Rule features:",len([c for c in rule_frame.columns if not c.startswith("target_")]))


## 8. Cost-bounded combination mining with candidate pruning

In [ ]:

def wilson(successes,total):
    if not total:return (np.nan,np.nan)
    z=1.95996398454;p=successes/total;d=1+z*z/total
    center=(p+z*z/(2*total))/d;margin=z*math.sqrt((p*(1-p)+z*z/(4*total))/total)/d
    return center-margin,center+margin

def mine_combinations(data):
    target="target_flag" if TARGET_MODE in ["binary","one_vs_rest"] else "target_class"
    features=[c for c in data.columns if c not in ["target_flag","target_class","target_ordinal"]]
    base=(data[target].mean() if target=="target_flag" else data[target].value_counts(normalize=True).to_dict())
    results=[]
    # Only feature sets whose individual features produced at least one qualifying segment are expanded.
    active_features=set()
    for length in range(1,MAX_RULE_LENGTH+1):
        feature_sets=combinations(features,length)
        level=[]
        for cols in feature_sets:
            if length>1 and not all(c in active_features for c in cols): continue
            if target=="target_flag":
                g=data.groupby(list(cols),dropna=False)[target].agg(records="count",class_count="sum").reset_index()
                g["target_class"]=str(POSITIVE_CLASS if TARGET_MODE=="binary" else FOCUS_CLASS)
                g["class_rate"]=g["class_count"]/g["records"];g["global_class_rate"]=base
                g["lift"]=g["class_rate"]/max(base,1e-9)
            else:
                g=data.groupby(list(cols),dropna=False)[target].value_counts().rename("class_count").reset_index()
                sup=data.groupby(list(cols),dropna=False).size().rename("records").reset_index();g=g.merge(sup,on=list(cols))
                g=g.rename(columns={target:"target_class"});g["class_rate"]=g["class_count"]/g["records"]
                g["global_class_rate"]=g["target_class"].map(base);g["lift"]=g["class_rate"]/g["global_class_rate"]
            g=g[(g.records>=MIN_SUPPORT_COUNT)&(g.class_count>=MIN_CLASS_COUNT)&(g.class_rate>=MIN_CONFIDENCE)&(g.lift>=MIN_LIFT)]
            if len(g) and length==1: active_features.update(cols)
            for _,r in g.iterrows():
                lo,hi=wilson(int(r.class_count),int(r.records))
                level.append({"rule_length":length,"rule":" AND ".join(f"{c} = {r[c]}" for c in cols),
                    "features":" | ".join(cols),"target_class":str(r.target_class),"records":int(r.records),
                    "class_count":int(r.class_count),"class_rate":float(r.class_rate),
                    "global_class_rate":float(r.global_class_rate),"lift":float(r.lift),"ci_low":lo,"ci_high":hi})
        if level:
            level=sorted(level,key=lambda r:(r["lift"]*r["class_rate"]*np.log1p(r["records"])),reverse=True)[:MAX_RULES_PER_LEVEL]
            results.extend(level)
    out=pd.DataFrame(results)
    if len(out):
        out["rule_score"]=out.lift*out.class_rate*np.log1p(out.records)*np.sqrt(out.class_count)
        out=out.sort_values(["rule_score","lift","records"],ascending=False).reset_index(drop=True)
    return out
combination_rules=mine_combinations(rule_frame)
display(combination_rules.head(150))


## 9. Optional FP-Growth on bounded sample

In [ ]:

association_rules_positive=pd.DataFrame()
if RUN_ASSOCIATION_RULES and len(rule_frame):
    target_exclusions={"target_flag","target_class","target_ordinal"}
    assoc_features=[c for c in rule_frame.columns if c not in target_exclusions][:MAX_ASSOCIATION_COLUMNS]
    parts=[]
    for c in assoc_features:
        parts.append(pd.get_dummies(rule_frame[c].astype(str).map(lambda x:f"{c}={x}"),prefix="",prefix_sep="").astype(bool))
    basket=pd.concat(parts,axis=1)
    if TARGET_MODE in ["binary","one_vs_rest"]:
        name=f"TARGET={POSITIVE_CLASS if TARGET_MODE=='binary' else FOCUS_CLASS}"
        basket[name]=rule_frame["target_flag"].astype(bool).values;target_items={name}
    else:
        td=pd.get_dummies(rule_frame["target_class"].astype(str).map(lambda x:f"TARGET={x}")).astype(bool)
        basket=pd.concat([basket,td],axis=1);target_items=set(td.columns)
    fi=fpgrowth(basket,min_support=ASSOCIATION_MIN_SUPPORT,use_colnames=True,max_len=ASSOCIATION_MAX_LEN)
    if len(fi):
        ar=association_rules(fi,metric="confidence",min_threshold=ASSOCIATION_MIN_CONFIDENCE)
        association_rules_positive=ar[ar.consequents.apply(lambda x:len(x)==1 and next(iter(x)) in target_items)&(ar.lift>=ASSOCIATION_MIN_LIFT)].copy()
        association_rules_positive["target_class"]=association_rules_positive.consequents.apply(lambda x:next(iter(x)).replace("TARGET=",""))
        association_rules_positive["rule"]=association_rules_positive.antecedents.apply(lambda x:" AND ".join(sorted(x)))+" => TARGET="+association_rules_positive.target_class
        association_rules_positive=association_rules_positive.sort_values(["lift","confidence","support"],ascending=False)
display(association_rules_positive[[c for c in ["rule","target_class","support","confidence","lift","leverage","conviction"] if c in association_rules_positive.columns]].head(150))


## 10. Optional bounded decision-tree model

In [ ]:

tree_rules=pd.DataFrame(); model_performance=pd.DataFrame()
if RUN_DECISION_TREE:
    model_features=[c for c in selected_num+selected_cat if c in df.columns]
    X=df[model_features].copy()
    y=df["target_flag"] if TARGET_MODE in ["binary","one_vs_rest"] else df["target_class"].astype(str)
    num=[c for c in model_features if pd.api.types.is_numeric_dtype(X[c])]
    cat=[c for c in model_features if c not in num]
    # Collapse categories before one-hot encoding to bound model width.
    for c in cat:
        top=X[c].astype("string").value_counts().head(MAX_MODEL_CATEGORIES).index
        X[c]=X[c].astype("string").where(X[c].astype("string").isin(top),"OTHER")
    prep=ColumnTransformer([
        ("num",Pipeline([("imputer",SimpleImputer(strategy="median"))]),num),
        ("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),
                         ("encoder",OneHotEncoder(handle_unknown="ignore",min_frequency=10))]),cat)
    ])
    model=DecisionTreeClassifier(max_depth=TREE_MAX_DEPTH,min_samples_leaf=TREE_MIN_SAMPLES_LEAF,class_weight="balanced",random_state=RANDOM_STATE)
    pipe=Pipeline([("prep",prep),("model",model)])
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=TEST_SIZE,stratify=y,random_state=RANDOM_STATE)
    pipe.fit(Xtr,ytr);pred=pipe.predict(Xte);proba=pipe.predict_proba(Xte)
    report=classification_report(yte,pred,output_dict=True,zero_division=0)
    model_performance=pd.DataFrame(report).T.reset_index().rename(columns={"index":"metric_or_class"})
    names=list(pipe.named_steps["prep"].get_feature_names_out());tree=pipe.named_steps["model"];classes=[str(x) for x in tree.classes_]
    rows=[]
    def walk(node,conds):
        if tree.tree_.feature[node]!=_tree.TREE_UNDEFINED:
            name=names[tree.tree_.feature[node]];thr=tree.tree_.threshold[node]
            walk(tree.tree_.children_left[node],conds+[f"{name} <= {thr:.4f}"])
            walk(tree.tree_.children_right[node],conds+[f"{name} > {thr:.4f}"])
        else:
            counts=tree.tree_.value[node][0];total=counts.sum();i=int(np.argmax(counts))
            rows.append({"rule":" AND ".join(conds),"weighted_records":float(total),
                         "predicted_class":classes[i],"predicted_class_rate":float(counts[i]/total) if total else 0})
    walk(0,[]);tree_rules=pd.DataFrame(rows).sort_values(["predicted_class_rate","weighted_records"],ascending=False)
display(model_performance)
display(tree_rules.head(100))


## 11. Lightweight holdout validation of top rules

In [ ]:

def apply_rule(data,rule):
    m=pd.Series(True,index=data.index)
    for cond in rule.split(" AND "):
        f,v=cond.split(" = ",1)
        if f not in data.columns:return pd.Series(False,index=data.index)
        m &= data[f].astype(str).eq(v)
    return m
validated_rules=pd.DataFrame()
if len(combination_rules):
    target="target_flag" if TARGET_MODE in ["binary","one_vs_rest"] else "target_class"
    train_idx,test_idx=train_test_split(rule_frame.index,test_size=TEST_SIZE,stratify=rule_frame[target],random_state=RANDOM_STATE)
    test=rule_frame.loc[test_idx];base=(test[target].mean() if target=="target_flag" else test[target].value_counts(normalize=True).to_dict())
    rows=[]
    for _,r in combination_rules.head(1000).iterrows():
        m=apply_rule(test,r.rule);n=int(m.sum());cls=str(r.target_class)
        if target=="target_flag":k=int(test.loc[m,target].sum()) if n else 0;b=base
        else:k=int((test.loc[m,target].astype(str)==cls).sum()) if n else 0;b=base.get(cls,np.nan)
        rate=k/n if n else np.nan
        rows.append({"rule":r.rule,"target_class":cls,"test_records":n,"test_class_count":k,
                     "test_class_rate":rate,"test_lift":rate/b if n and b and b>0 else np.nan})
    validated_rules=pd.DataFrame(rows).sort_values(["test_lift","test_records"],ascending=False)
display(validated_rules.head(150))


## 12. Final rules, Delta outputs, and downloadable summaries

In [ ]:

final_rules=combination_rules.copy()
if len(final_rules) and len(validated_rules):
    final_rules=final_rules.merge(validated_rules,on=["rule","target_class"],how="left")
if len(final_rules):
    final_rules["rule_id"]=[f"R{i:05d}" for i in range(1,len(final_rules)+1)]
    final_rules["recommended_action"]=np.select(
        [final_rules.get("test_lift",pd.Series(index=final_rules.index,dtype=float)).ge(2)&final_rules.get("test_records",0).ge(20),
         final_rules.lift.ge(2),final_rules.lift.ge(1.5)],
        ["High-priority specialist review","Enhanced validation","Additional automated checks"],default="Monitor")

def safe_table_name(s): return re.sub(r"[^a-zA-Z0-9_]","_",s)

if OUTPUT_MODE in ["DELTA","BOTH"] and SPARK_AVAILABLE:
    outputs={"profile":profile,"numeric_summary":numeric_summary,"combination_rules":combination_rules,
             "association_rules":association_rules_positive,"tree_rules":tree_rules,
             "validated_rules":validated_rules,"final_business_rules":final_rules,"model_performance":model_performance}
    for suffix,pdf in outputs.items():
        if pdf is not None and len(pdf):
            table=f"{OUTPUT_DELTA_SCHEMA}.{safe_table_name(OUTPUT_TABLE_PREFIX+'_'+suffix)}"
            spark.createDataFrame(pdf.replace({np.nan:None})).write.mode("overwrite").option("overwriteSchema","true").saveAsTable(table)
            print("Wrote",table)

if OUTPUT_MODE in ["FILES","BOTH"]:
    os.makedirs(OUTPUT_FOLDER,exist_ok=True)
    path=os.path.join(OUTPUT_FOLDER,OUTPUT_EXCEL)
    # Excel is a summary channel; cap each sheet to avoid huge driver-side files.
    with pd.ExcelWriter(path,engine="openpyxl") as w:
        profile.head(200000).to_excel(w,"Data_Profile",index=False)
        numeric_summary.head(200000).to_excel(w,"Numeric_EDA",index=False)
        combination_rules.head(200000).to_excel(w,"Combination_Rules",index=False)
        association_rules_positive.head(200000).to_excel(w,"Association_Rules",index=False)
        tree_rules.head(200000).to_excel(w,"Tree_Rules",index=False)
        validated_rules.head(200000).to_excel(w,"Validated_Rules",index=False)
        final_rules.head(200000).to_excel(w,"Final_Business_Rules",index=False)
        bin_metadata.to_excel(w,"Bin_Metadata",index=False)
    print("Created",path)

if sdf is not None and ENABLE_CACHE and UNPERSIST_AT_END:
    sdf.unpersist(blocking=False)

display(final_rules.head(150))



## 13. Performance and cost guidance

- Project only required columns through `SOURCE_COLUMNS`; this is the largest avoidable I/O saving.
- Apply `SOURCE_FILTER` before profiling, caching, or sampling.
- Use Delta tables and Photon-compatible SQL expressions; the notebook avoids Python UDFs.
- Full-volume work is restricted to compact Spark aggregations. FP-Growth, exhaustive rules, and scikit-learn run only on the bounded driver sample.
- Increase `MAX_DRIVER_ROWS` only when the driver has sufficient memory.
- Keep `MAX_RULE_FEATURES` and `MAX_RULE_LENGTH` conservative. Rule combinations grow rapidly.
- High-cardinality identifiers, free text, and near-unique columns are excluded from rules and one-hot encoding.
- `percentile_approx` and `approx_count_distinct` reduce shuffle and compute cost.
- Use `DISK_ONLY` caching for a reused large frame; disable caching when the source is scanned only once.
- Persist final reusable results as Delta. Excel is intentionally limited to compact review outputs.
- For recurring production runs, schedule the notebook as a Databricks job and use job clusters or serverless jobs rather than an always-on interactive cluster.
